In [16]:
import pickle
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Final
import sys
import os


CURRENT_DIR: Final[Path] = Path(os.getcwd())
PROJECT_DIR: Final[Path] = CURRENT_DIR.parent
sys.path.append(str(PROJECT_DIR))


In [17]:
from log_parser import LogParser


In [18]:
pd.set_option("display.float_format", lambda x: "%.3f" % x)

# Custom Statistics

## Config

In [ ]:
CUSTOM_KEYWORDS: Final[list[str]] = [
    "[FREE_MOVE_PREDICTOR_SUM]",
    "[LANE_SEQUENCE_PREDICTOR_SUM]",
    "[JUNCTION_PREDICTOR_SUM]",
    "[LEARNING_BASED_PREDICTOR_SUM]",
]

log_path = "/Projects/gpal/eka_project/log/policy_planner/2025-05-26"


## Custom Functions

In [23]:
def parse_custom_logs(output_path: str | Path, lines: list[str], keywords: list[str]):
    statistics = {keyword: [] for keyword in keywords}
    for line in lines:
        if "ms" not in line:
            continue

        for keyword in keywords:
            if keyword not in line:
                continue

            try:
                line_split = line.split()
                duration = float(line_split[-2])
                number = float(line_split[-3].split("(")[-1].split(")")[0])
            except ValueError:
                continue

            statistics[keyword].append((number, duration))

    # remove not exist keywords
    for keyword in keywords:
        if statistics[keyword]:
            continue
        statistics.pop(keyword)

    with open(output_path, "wb") as fw:
        pickle.dump(statistics, fw)


def count_numbers(x):
    return int(np.log10(x)) + 1

## Read Logs

In [24]:
log_lines = LogParser(log_path).lines

stat_result_name = "custom_stat_result.pkl"
database_path = CURRENT_DIR / stat_result_name
parse_custom_logs(database_path, log_lines, CUSTOM_KEYWORDS)


## Custom Statistics

In [25]:
with open(database_path, "rb") as f:
    raw_data = pickle.load(f)

max_keyword_len = max([len(key) for key in raw_data.keys()])
max_num_len = max([count_numbers(np.array(value)[:, 0].sum()) for value in raw_data.values()])
max_duration_len = max([count_numbers(np.array(value)[:, 1].sum()) for value in raw_data.values()]) + 3
for key, value in raw_data.items():
    stats = np.array(value)
    total_num = stats[:, 0].sum(dtype=np.int64)
    total_duration = stats[:, 1].sum(dtype=np.float64)
    print(
        f"{key:<{max_keyword_len}}: average({total_duration / total_num:.2f} ms), agent_num({total_num:>{max_num_len}}), duration({total_duration:>{max_duration_len}.2f} ms)"
    )


[FREE_MOVE_PREDICTOR_SUM]    : average(0.01 ms), agent_num(13900), duration(182.02 ms)
[LANE_SEQUENCE_PREDICTOR_SUM]: average(0.05 ms), agent_num(13253), duration(643.54 ms)
[JUNCTION_PREDICTOR_SUM]     : average(0.03 ms), agent_num( 1644), duration( 47.35 ms)
